In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [7]:
# Define the reference function
def custom_function(x):
    return np.sin(x) + np.cos(x)

In [8]:
class ArrayNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(ArrayNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.to(self.device)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [9]:
# Example usage:
input_size = 5
hidden_size = 10
output_size = 5

# Create instance of the network
net = ArrayNet(input_size, hidden_size, output_size)

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.SGD(net.parameters(), lr=0.01)

In [10]:

# Generate some dummy data using the user-defined function
numOfdata = 100
input_data = torch.randn(numOfdata, input_size, device=net.device)  # 100 samples, 5 features each
target_data = torch.zeros(numOfdata, output_size, device=net.device)
for i in range(numOfdata):
    target_data[i] = torch.from_numpy(custom_function(input_data[i].cpu().numpy())).to(net.device)

In [11]:
# Train the network
num_epochs = 5000
for epoch in range(num_epochs):
    # Forward pass
    outputs = net(input_data)
    loss = criterion(outputs, target_data)
    
    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [100/5000], Loss: 0.7027
Epoch [200/5000], Loss: 0.5443
Epoch [300/5000], Loss: 0.4701
Epoch [400/5000], Loss: 0.4224
Epoch [500/5000], Loss: 0.3845
Epoch [600/5000], Loss: 0.3518
Epoch [700/5000], Loss: 0.3234
Epoch [800/5000], Loss: 0.2989
Epoch [900/5000], Loss: 0.2779
Epoch [1000/5000], Loss: 0.2597
Epoch [1100/5000], Loss: 0.2443
Epoch [1200/5000], Loss: 0.2311
Epoch [1300/5000], Loss: 0.2199
Epoch [1400/5000], Loss: 0.2102
Epoch [1500/5000], Loss: 0.2021
Epoch [1600/5000], Loss: 0.1951
Epoch [1700/5000], Loss: 0.1889
Epoch [1800/5000], Loss: 0.1833
Epoch [1900/5000], Loss: 0.1784
Epoch [2000/5000], Loss: 0.1737
Epoch [2100/5000], Loss: 0.1694
Epoch [2200/5000], Loss: 0.1660
Epoch [2300/5000], Loss: 0.1627
Epoch [2400/5000], Loss: 0.1596
Epoch [2500/5000], Loss: 0.1566
Epoch [2600/5000], Loss: 0.1536
Epoch [2700/5000], Loss: 0.1508
Epoch [2800/5000], Loss: 0.1481
Epoch [2900/5000], Loss: 0.1456
Epoch [3000/5000], Loss: 0.1430
Epoch [3100/5000], Loss: 0.1406
Epoch [3200/5000]

In [13]:
# Assuming 'net' is the trained network from the previous code snippet

# Generate new input data for prediction
new_input_data = torch.randn(1, input_size, device=net.device)  # 1 sample, 5 features each

# generate test
test_res = custom_function(new_input_data[0].cpu().numpy())

print(test_res)

# Put the network in evaluation mode
net.eval()

[0.8029429  1.4142122  1.2962377  0.02279097 1.3867171 ]


ArrayNet(
  (fc1): Linear(in_features=5, out_features=10, bias=True)
  (fc2): Linear(in_features=10, out_features=5, bias=True)
)

In [14]:
# Make prediction
with torch.no_grad():
    prediction = net(new_input_data)

print("Prediction:", prediction)

Prediction: tensor([[0.6078, 1.1021, 0.9676, 0.1080, 1.0734]], device='cuda:0')


In [15]:
# tracing
traced_script_module = torch.jit.trace(net, new_input_data)

In [16]:
# Assuming 'net' is the trained network from the previous code snippet

# Specify the file path where you want to save the model
if net.device.type == 'cuda':
    model_path = "array_net_model_cuda.pt"
    torch.jit.save(traced_script_module, model_path)

    # convert the cuda model to cpu, and save it again.
    traced_script_module_cpu = traced_script_module.to('cpu')
    model_path_cpu = "array_net_model_cpu.pt"
    torch.jit.save(traced_script_module_cpu, model_path_cpu)
    print(f"Model saved to {model_path} and {model_path_cpu}")
else:
    model_path = "array_net_model_cpu.pt"
    torch.jit.save(traced_script_module, model_path)    
    print(f"Model saved to {model_path}")

Model saved to array_net_model_cuda.pt and array_net_model_cpu.pt
